# 10-Point Inspection — NHANES 2005 Accelerometry
### Biomedical Informatics · Accelerometer-Projecter

| Point | Check | Command |
|-------|-------|---------|
| 1 | Shape | `df.shape` |
| 2 | Columns | `df.columns` |
| 3 | Data Types | `df.dtypes` |
| 4 | First Look | `df.head()` |
| 5 | Last Look | `df.tail()` |
| 6 | Memory | `df.memory_usage()` |
| 7 | Missing | `df.isnull().sum()` |
| 8 | Duplicates | `df.duplicated()` |
| 9 | Statistics | `df.describe()` |
| 10 | Unique Values | `df.nunique()` |

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

MASTER_PATH = "/Users/graysongrabois/Documents/biomedical-informatics/Opioid Project/2005_master.csv"
AGG_PATH    = "/Users/graysongrabois/Documents/biomedical-informatics/Opioid Project/2005_patient_aggregated.csv"

print("Loading agg...")
agg = pd.read_csv(AGG_PATH)
print(f"  agg: {agg.shape}")

print("Loading master (large, ~30s)...")
master = pd.read_csv(MASTER_PATH)
print(f"  master: {master.shape}")
print("Done.")

Loading agg...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/graysongrabois/Documents/biomedical-informatics/Opioid Project/2005_patient_aggregated.csv'

---
## Point 1 — Shape
How many rows and columns in each file?

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    rows, cols = df.shape
    print(f"{name}")
    print(f"  Rows         {rows:>15,}")
    print(f"  Columns      {cols:>15,}")
    print(f"  Total cells  {rows * cols:>15,}")
    print()

---
## Point 2 — Column Names
What features do we have, and which are shared vs unique to each file?

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    print(f"{name} ({len(df.columns)} columns):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2}. {col}")
    print()

print("Only in AGG   :", sorted(set(agg.columns)    - set(master.columns)))
print("Only in MASTER:", sorted(set(master.columns) - set(agg.columns)))

---
## Point 3 — Data Types
How is each column stored?

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    print(f"=== {name} ===")
    print(df.dtypes.to_string())
    print(f"\n  Summary: {df.dtypes.value_counts().to_dict()}")
    print()

---
## Point 4 — First Look

In [ ]:
print("=== MASTER — first 5 rows ===")
display(master.head())
print("\n=== AGG — first 5 rows ===")
display(agg.head())

---
## Point 5 — Last Look

In [ ]:
print("=== MASTER — last 5 rows ===")
display(master.tail())
print("\n=== AGG — last 5 rows ===")
display(agg.tail())

---
## Point 6 — Memory Usage

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    mem = df.memory_usage(deep=True)
    print(f"=== {name} — {mem.sum() / 1e6:.1f} MB total ===")
    for col, val in (mem / 1e6).items():
        if col != 'Index':
            print(f"  {col:<25} {val:.2f} MB")
    print()

---
## Point 7 — Missing Values

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    miss = df.isnull().sum()
    pct  = (miss / len(df) * 100).round(2)
    report = pd.DataFrame({'missing': miss, 'pct': pct})
    report = report[report['missing'] > 0].sort_values('pct', ascending=False)
    print(f"=== {name} ({len(df):,} rows) ===")
    if report.empty:
        print("  No missing values.")
    else:
        print(report.to_string())
    print()

---
## Point 8 — Duplicates

In [ ]:
# Full-row duplicates
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    n = df.duplicated().sum()
    print(f"{name} full-row duplicates: {n:,} ({n/len(df)*100:.2f}%)")

print()

# Composite key check for MASTER
n = master.duplicated(subset=['participant_id','day','hour','minute']).sum()
print(f"MASTER duplicate (id, day, hour, minute): {n:,}")

# Primary key check for AGG
n = agg.duplicated(subset='participant_id').sum()
print(f"AGG duplicate participant_id: {n:,}")

---
## Point 9 — Descriptive Statistics

In [ ]:
print("=== AGG — describe (transposed) ===")
display(agg.describe().T)

print("\n=== MASTER — activity columns ===")
display(master[['intensity','steps']].describe().T)

print("\n=== MASTER — demographics (one row per patient) ===")
demo = ['age','sex','race','poverty_ratio','bmi','systolic_bp','diastolic_bp',
        'weight_kg','height_cm','waist_cm']
display(master.drop_duplicates('participant_id')[demo].describe().T)

---
## Point 10 — Unique Values

In [ ]:
for df, name in [(master, 'MASTER'), (agg, 'AGG')]:
    u = df.nunique().sort_values()
    print(f"=== {name} unique values per column ===")
    print(u.to_string())
    print(f"\n  Binary (2)     : {u[u==2].index.tolist()}")
    print(f"  Low card (3-10): {u[(u>2)&(u<=10)].index.tolist()}")
    print(f"  High card (>10): {u[u>10].index.tolist()}")
    print()

---
## Final Checklist

In [ ]:
print("=" * 52)
print("  FINAL DATA CHECKLIST")
print("=" * 52)

# ----------------------------------------------------------
# 1. File sizes
# ----------------------------------------------------------
print("\n[ ] File Sizes")
print(f"    master   {len(master):>12,} rows  x  {master.shape[1]} cols")
print(f"    agg      {len(agg):>12,} rows  x  {agg.shape[1]} cols")

# ----------------------------------------------------------
# 2. Patient key alignment
# ----------------------------------------------------------
print("\n[ ] Patient Key Alignment")
m_ids = set(master['participant_id'].unique())
a_ids = set(agg['participant_id'].unique())
unmatched = m_ids.symmetric_difference(a_ids)
print(f"    Unique IDs — master: {len(m_ids):,}   agg: {len(a_ids):,}")
if unmatched:
    print(f"    UNMATCHED: {sorted(unmatched)}")
else:
    print("    No unmatched patients   -->   PASS")

# ----------------------------------------------------------
# 3. Remaining missing values
# ----------------------------------------------------------
print("\n[ ] Remaining Missing Values (agg)")
miss = agg.isnull().sum()
miss = miss[miss > 0]
if miss.empty:
    print("    None   -->   PASS")
else:
    for col, n in miss.items():
        print(f"    {col:<22}  {n:>4} missing  ({n/len(agg)*100:.1f}%)")

# ----------------------------------------------------------
# 4. Physiological outlier check
# ----------------------------------------------------------
print("\n[ ] Physiological Outlier Check (agg)")
BOUNDS = {
    'age'         : (18, 120),
    'bmi'         : (10,  80),
    'systolic_bp' : (50, 300),
    'diastolic_bp': (20, 200),
    'weight_kg'   : (20, 300),
    'height_cm'   : (100, 230),
    'waist_cm'    : (40, 200),
}
any_bad = False
for col, (lo, hi) in BOUNDS.items():
    if col not in agg.columns:
        continue
    sub  = agg[col].dropna()
    nbad = ((sub < lo) | (sub > hi)).sum()
    if nbad:
        print(f"    {col:<22}  {nbad} values outside [{lo}, {hi}]  "
              f"(min={sub.min():.1f}, max={sub.max():.1f})")
        any_bad = True
if not any_bad:
    print("    All columns within physiological bounds   -->   PASS")

# Accelerometry bounds (master)
print("\n[ ] Accelerometry Bounds (master)")
checks = [
    ('intensity < 0',       (master['intensity'] < 0).sum()),
    ('steps < 0',           (master['steps'] < 0).sum()),
    ('steps > 200/min',     (master['steps'] > 200).sum()),
    ('intensity == 32767',  (master['intensity'] == 32767).sum()),
]
accel_clean = True
for label, n in checks:
    flag = '  WARN' if n > 0 else ''
    print(f"    {label:<28}  {n:,}{flag}")
    if n > 0:
        accel_clean = False
if accel_clean:
    print("    All accelerometry checks clean   -->   PASS")

# ----------------------------------------------------------
# 5. Avg daily intensity: hospitalized vs not
# ----------------------------------------------------------
print("\n[ ] Avg Daily Intensity — Hospitalized vs Not")
daily = (master.groupby(['participant_id','day'])['intensity']
               .mean()
               .reset_index()
               .merge(agg[['participant_id','hospitalized']], on='participant_id'))
grp = daily.groupby('hospitalized')['intensity'].mean()
not_h = grp.get(0, float('nan'))
hosp  = grp.get(1, float('nan'))
print(f"    Not hospitalized  {not_h:>7.1f}  avg counts/day")
print(f"    Hospitalized      {hosp:>7.1f}  avg counts/day")
print(f"    Difference        {not_h - hosp:>7.1f}  ({'not-hosp more active' if not_h > hosp else 'hosp more active'})")

print("\n" + "=" * 52)